# Analysis: do LLMs know when they're wrong?

Loads `results/raw/*.jsonl`, builds the results table, runs paired comparisons, and draws the three figures. The same code runs headless via `python src/analyze.py`.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
from metrics import load_results, results_table
from analyze import comparisons
import plots

df = load_results()
df.groupby(['model_key', 'condition']).agg(rows=('question_id', 'size'), parse_errors=('parse_error', 'sum'))

## Results table

Accuracy and type-2 AUROC have 95% bootstrap intervals (1,000 resamples). `critic_fixed` / `critic_broke` count answers condition C changed from wrong to right / right to wrong.

In [ ]:
table = results_table(df)
table.round(3)

## Are the differences real?

Paired bootstrap on the same questions. If `ci_excludes_0` is False, the difference could be noise at n=200.

In [ ]:
comparisons(df).round(3)

## Figure 1: reliability diagram

Points on the diagonal = calibrated. Points below = overconfident.

In [ ]:
plots.reliability_diagram(df, '../figures/reliability_diagram.png');

## Figure 2: confidence on right vs wrong answers

The more the two distributions separate, the higher the metacognitive sensitivity.

In [ ]:
plots.confidence_by_correctness(df, '../figures/confidence_by_correctness.png');

## Figure 3: risk-coverage

Moving right, the model defers its least confident answers to a human. A curve that rises means confidence is useful for deciding what to defer.

In [ ]:
plots.risk_coverage_plot(df, '../figures/risk_coverage.png');

## How often do models say 90+?

LLMs often report very high confidence on almost everything. If so, that's a finding.

In [ ]:
ok = df[~df['parse_error'].astype(bool)]
ok.assign(conf_90_plus=ok['confidence'] >= 0.9).groupby(['model_key', 'condition'])['conf_90_plus'].mean().unstack().round(2)

## Self-consistency: do the samples' own verbal confidences track agreement?

In [ ]:
b = ok[ok['condition'] == 'B']
b.groupby(['model_key', 'confidence'])['mean_verbalized_confidence'].agg(['mean', 'size']).round(2)